In [ ]:
import os, json
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/PACL_Project"
PHASE3_DIR = f"{PROJECT_DIR}/phase3_outputs"
PHASE4_DIR = f"{PROJECT_DIR}/phase4_outputs"
os.makedirs(PHASE4_DIR, exist_ok=True)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_records = load_jsonl(f"{PHASE3_DIR}/train.jsonl")
train_texts = [r["text"] for r in train_records]
n_train_poets = len(set(r["poet"] for r in train_records))
print(f"train_records (از phase3_outputs، اصلاح‌شده): {len(train_records)} نمونه / {n_train_poets} شاعر")
assert n_train_poets == 150, (
    f"❌ انتظار داشتیم دقیقاً ۱۵۰ شاعر در train (فایلِ اصلاح‌شدهِ phase3_outputs)، شد {n_train_poets}. "
    "قبل از ادامه مسیرِ PHASE3_DIR را بررسی کنید (این همان فایلی است که Ph3V2ponzdah.ipynb هم ازش استفاده می‌کند)."
)

test_open_records = load_jsonl(f"{PHASE3_DIR}/test_open.jsonl")
print("کلِ رکوردهایِ test_open:", len(test_open_records))

k1_indices = [i for i, r in enumerate(test_open_records) if r.get("k") == 1]
print(f"تعدادِ رکوردهایِ Open با k=1: {len(k1_indices)}")

assert len(k1_indices) > 0, "❌ هیچ رکوردِ k=1 در test_open پیدا نشد — نامِ فیلدِ 'k' را بررسی کنید."


In [ ]:
import numpy as np

pacl_test_open_emb = np.load(f"{PHASE3_DIR}/test_open_embeddings_byt5base.npy")
print("شکلِ embeddingِ PACL برایِ کلِ test_open:", pacl_test_open_emb.shape)
assert pacl_test_open_emb.shape[0] == len(test_open_records), (
    "❌ تعدادِ ردیف‌هایِ embeddingِ PACL با تعدادِ رکوردهایِ test_open یکی نیست — این دو باید از یک فایل و با یک ترتیب ساخته شده باشند."
)

pacl_k1_emb = pacl_test_open_emb[k1_indices]
poets_k1 = [test_open_records[i]["poet"] for i in k1_indices]
poem_ids_k1 = [test_open_records[i]["poem_id"] for i in k1_indices]
texts_k1 = [test_open_records[i]["text"] for i in k1_indices]
print("شکلِ embeddingِ PACL برایِ زیرمجموعهِ Open k=1:", pacl_k1_emb.shape)
print("تعدادِ شاعرانِ یکتا در این زیرمجموعه:", len(set(poets_k1)))


In [ ]:
import time
from sklearn.feature_extraction.text import TfidfVectorizer

t0 = time.time()
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), max_features=50000, sublinear_tf=True)
vectorizer.fit(train_texts)
print(f"TF-IDF رویِ Train فیت شد. اندازهٔ واژگان: {len(vectorizer.vocabulary_)}  ({time.time()-t0:.1f}s)")

X_open_k1 = vectorizer.transform(texts_k1)
print("شکلِ ماتریسِ TF-IDF برایِ زیرمجموعهِ Open k=1:", X_open_k1.shape, f"  ({time.time()-t0:.1f}s)")


In [ ]:
import numpy as np
import scipy.sparse as sp
from sklearn.preprocessing import normalize as sk_normalize
import time

def evaluate_retrieval_top1_per_query(query_vecs, query_poets, query_poem_ids, query_texts,
                                       gallery_vecs, gallery_poets, gallery_poem_ids, gallery_texts,
                                       exclude_same_poem=True, batch_size=500):
    def l2_normalize(mat):
        if sp.issparse(mat):
            return sk_normalize(mat, norm="l2", axis=1)
        mat = np.asarray(mat, dtype=np.float32)
        return mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12)

    q = l2_normalize(query_vecs)
    g = l2_normalize(gallery_vecs)
    query_poets = np.asarray(query_poets); gallery_poets = np.asarray(gallery_poets)
    query_poem_ids = np.asarray(query_poem_ids); gallery_poem_ids = np.asarray(gallery_poem_ids)
    gallery_texts = np.asarray(gallery_texts, dtype=object)

    n_queries = q.shape[0]
    out = [None] * n_queries

    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)
        batch_q = q[start:end]
        sims = batch_q @ g.T
        if sp.issparse(sims):
            sims = sims.toarray()
        sims = np.asarray(sims, dtype=np.float32)

        if exclude_same_poem:
            batch_poem_ids = query_poem_ids[start:end]
            same_poem_mask = batch_poem_ids[:, None] == gallery_poem_ids[None, :]
            sims = np.where(same_poem_mask, -np.inf, sims)

        top1_idx = np.argmax(sims, axis=1)
        rows = np.arange(end - start)
        top1_sim = sims[rows, top1_idx]

        for i in range(end - start):
            qi = start + i
            if np.isneginf(top1_sim[i]):
                continue
            gi = int(top1_idx[i])
            pred_poet = str(gallery_poets[gi])
            true_poet = str(query_poets[qi])
            out[qi] = {
                "query_idx": int(qi),
                "query_text": str(query_texts[qi]),
                "true_poet": true_poet,
                "pred_poet": pred_poet,
                "correct": pred_poet == true_poet,
                "retrieved_poem_id": gallery_poem_ids[gi].item() if hasattr(gallery_poem_ids[gi], "item") else gallery_poem_ids[gi],
                "retrieved_text": str(gallery_texts[gi]),
                "top1_similarity": float(top1_sim[i]),
            }
    return out

t0 = time.time()
print("در حالِ اجرایِ ارزیابیِ per-query برایِ PACL رویِ Open k=1 ...")
pacl_results = evaluate_retrieval_top1_per_query(
    pacl_k1_emb, poets_k1, poem_ids_k1, texts_k1,
    pacl_k1_emb, poets_k1, poem_ids_k1, texts_k1,
    exclude_same_poem=True,
)
print(f"PACL تمام شد. ({time.time()-t0:.1f}s)")

print("در حالِ اجرایِ ارزیابیِ per-query برایِ TF-IDF رویِ Open k=1 ...")
tfidf_results = evaluate_retrieval_top1_per_query(
    X_open_k1, poets_k1, poem_ids_k1, texts_k1,
    X_open_k1, poets_k1, poem_ids_k1, texts_k1,
    exclude_same_poem=True,
)
print(f"TF-IDF تمام شد. ({time.time()-t0:.1f}s)")

pacl_valid = [r for r in pacl_results if r is not None]
tfidf_valid = [r for r in tfidf_results if r is not None]
pacl_recall1 = sum(r["correct"] for r in pacl_valid) / len(pacl_valid)
tfidf_recall1 = sum(r["correct"] for r in tfidf_valid) / len(tfidf_valid)
print(f"\nچکِ صحت — Recall@1 محاسبه‌شده در این نوت‌بوک:")
print(f"  PACL:   {pacl_recall1:.4f}  (n_valid={len(pacl_valid)})  <-- باید نزدیکِ عددِ رسمیِ قبلی 0.3129 باشد")
print(f"  TF-IDF: {tfidf_recall1:.4f}  (n_valid={len(tfidf_valid)})  <-- باید نزدیکِ عددِ رسمیِ قبلی 0.3577 باشد")
print("اگر هرکدام از این دو عدد فاصلهٔ زیادی (بیش از چند صدم) با عددِ رسمی داشت، قبل از ادامه به من خبر بده.")


In [ ]:
import random

candidate_indices = [
    i for i in range(len(poets_k1))
    if pacl_results[i] is not None and tfidf_results[i] is not None
    and (not pacl_results[i]["correct"]) and tfidf_results[i]["correct"]
]
print(f"تعدادِ queryهایی که PACL غلط و TF-IDF درست است: {len(candidate_indices)} از {len(poets_k1)} queryِ Open k=1")

random.seed(42)
N_SAMPLES = 15
sample_indices = sorted(random.sample(candidate_indices, min(N_SAMPLES, len(candidate_indices))))
print(f"تعدادِ نمونه‌هایِ انتخاب‌شده برایِ بررسیِ دستی: {len(sample_indices)}\n")

examples = []
for rank, i in enumerate(sample_indices, start=1):
    p = pacl_results[i]
    t = tfidf_results[i]
    ex = {
        "rank": rank,
        "query_idx": i,
        "query_poem_id": poem_ids_k1[i],
        "query_text": texts_k1[i],
        "true_poet": p["true_poet"],
        "pacl_wrong_pred_poet": p["pred_poet"],
        "pacl_retrieved_text": p["retrieved_text"],
        "pacl_top1_similarity": p["top1_similarity"],
        "tfidf_correct_pred_poet": t["pred_poet"],
        "tfidf_retrieved_text": t["retrieved_text"],
        "tfidf_top1_similarity": t["top1_similarity"],
    }
    examples.append(ex)
    print(f"=== نمونهٔ {rank} (query_idx={i}, poem_id={ex['query_poem_id']}) ===")
    print(f"متنِ query (شاعرِ واقعی: {ex['true_poet']}):")
    print(f"  {ex['query_text']}")
    print(f"PACL اشتباهاً به «{ex['pacl_wrong_pred_poet']}» نسبت داد (sim={ex['pacl_top1_similarity']:.3f}), متنِ بازیابی‌شده:")
    print(f"  {ex['pacl_retrieved_text']}")
    print(f"TF-IDF درست به «{ex['tfidf_correct_pred_poet']}» نسبت داد (sim={ex['tfidf_top1_similarity']:.3f}), متنِ بازیابی‌شده:")
    print(f"  {ex['tfidf_retrieved_text']}")
    print()


In [ ]:
import json as _json

summary = {
    "scenario": "Open k=1",
    "n_total_queries": len(poets_k1),
    "n_pacl_wrong_and_tfidf_correct": len(candidate_indices),
    "pacl_recall_at_1": pacl_recall1,
    "tfidf_recall_at_1": tfidf_recall1,
    "n_examples_sampled": len(examples),
    "sample_seed": 42,
    "examples": examples,
}

out_path = f"{PHASE4_DIR}/step3_k1_error_analysis.json"
with open(out_path, "w", encoding="utf-8") as f:
    _json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"✅ ذخیره شد در {out_path}")
print(f"تعدادِ نمونه‌هایِ ذخیره‌شده: {len(examples)}")
